In [1]:
def preprocess_burst_mimo(file_path):
    print("🔄 데이터 전처리 중...")
    data = np.load(file_path, allow_pickle=True)
    frequencies = np.linspace(-BANDWIDTH/2, BANDWIDTH/2, 64)
    
    X_list = [] # 입력: 과거 Burst (혹은 Burst 내 앞부분)
    Y_list = [] # 정답: 미래 Burst (혹은 Burst 내 뒷부분)
    
    # 논문의 예측 시나리오 가정:
    # "한 Burst 내에서 앞의 10개 스냅샷을 보고, 뒤의 10개 스냅샷 예측" 
    # (매우 짧은 시간 예측, Interpolation 성격)
    input_len = 10
    pred_len = 10
    
    for burst in tqdm(data):
        a_list = burst['a']
        tau_list = burst['tau']
        
        # Burst -> H 행렬 변환
        h_burst = []
        for i in range(len(a_list)):
            a_raw = a_list[i]
            tau_raw = tau_list[i]
            
            if a_raw.size == 0:
                h_burst.append(np.zeros((8, 24, 64), dtype=np.complex64))
                continue
                
            # 차원 정리: [Rx(1), RxAnt(8), Tx(3), TxAnt(8), Paths] -> [8, 24, Paths]
            # Tx 차원(3)과 TxAnt(8)을 합쳐서 24로 만듭니다. (CNN 이미지의 너비 역할)
            
            # a_raw shape: (1, 8, 3, 8, Paths)
            a_sq = np.squeeze(a_raw) # (8, 3, 8, Paths)
            a_flat = a_sq.reshape(8, 24, -1) # (8, 24, Paths)
            
            # tau shape: (1, 3, Paths) -> Broadcasting 필요
            tau_sq = np.squeeze(tau_raw) # (3, Paths)
            tau_flat = np.repeat(tau_sq[:, np.newaxis, :], 8, axis=1) # (3, 8, Paths)
            tau_flat = tau_flat.reshape(24, -1) # (24, Paths)
            tau_final = np.broadcast_to(tau_flat[np.newaxis, ...], (8, 24, tau_flat.shape[-1]))
            
            # 주파수 응답 계산
            # exp(-j 2pi f tau)
            phase = -1j * 2 * np.pi * tau_final[..., None] * frequencies[None, None, None, :]
            h_val = np.sum(a_flat[..., None] * np.exp(phase), axis=-2) # Sum paths
            h_burst.append(h_val) # (8, 24, 64)
            
        h_burst = np.array(h_burst) # (30, 8, 24, 64)
        
        # 슬라이딩 윈도우 생성 (Burst 내부에서)
        if len(h_burst) >= input_len + pred_len:
            for t in range(len(h_burst) - input_len - pred_len + 1):
                X_list.append(h_burst[t : t+input_len])
                Y_list.append(h_burst[t+input_len : t+input_len+pred_len])

    return np.array(X_list), np.array(Y_list)

# 데이터셋 생성 (메모리 주의: 필요시 배치 제너레이터 사용 권장)
# X, Y = preprocess_burst_mimo('channel_history_burst_mimo_3bs.npy')

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input

def build_paper_cnn_lstm(input_steps=10, pred_steps=10):
    # 입력 Shape: (Time, Rx, Tx, Freq*2) -> 복소수를 2채널로
    inputs = Input(shape=(input_steps, 8, 24, 128))
    
    # 1. Spatial Feature Extraction (CNN)
    # TimeDistributed: 시간축(Time)을 유지한 채 각 프레임에 CNN 적용
    x = layers.TimeDistributed(layers.Conv2D(64, (3,3), padding='same', activation='relu'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.Conv2D(128, (3,3), padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    
    # (Time, 8, 24, 128) -> Pooling -> (Time, 4, 12, 128)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)
    
    # Flatten: 이미지를 벡터로 (Time, 4*12*128) = (Time, 6144)
    x = layers.TimeDistributed(layers.Flatten())(x)
    
    # 2. Temporal Feature Extraction (LSTM)
    # Encoder
    x = layers.LSTM(512, return_sequences=False)(x) # 마지막 상태만 전달
    
    # Decoder (Future Generation)
    x = layers.RepeatVector(pred_steps)(x) # 예측할 시간만큼 복제
    x = layers.LSTM(512, return_sequences=True)(x)
    
    # 3. Output Reconstruction
    # (Time, 512) -> (Time, 8*24*128)
    x = layers.TimeDistributed(layers.Dense(8 * 24 * 128))(x)
    outputs = layers.Reshape((pred_steps, 8, 24, 128))(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Paper_CNN_LSTM")
    return model

# 멀티 GPU 학습 설정 (데이터 병렬 처리)
strategy = tf.distribute.MirroredStrategy()
print(f"✅ 학습 장치 수: {strategy.num_replicas_in_sync}")

with strategy.scope():
    model = build_paper_cnn_lstm()
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    model.summary()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
✅ 학습 장치 수: 2


Model: "Paper_CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10, 8, 24, 128) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_14             │ (None, 10, 8, 24, 64)  │        73,792 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_15             │ (None, 10, 8, 24, 64)  │           256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_16             │ (None, 10, 8, 24, 128) │        73,856 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_17             │ (None, 10, 8, 24, 128) │           512 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_18             │ (None, 10, 4, 12, 128) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_19             │ (None, 10, 6144)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 512)            │    13,633,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_2 (RepeatVector)  │ (None, 10, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 10, 512)        │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_20             │ (None, 10, 24576)      │    12,607,488 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 10, 8, 24, 128) │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,488,640 (108.68 MB)

 Trainable params: 28,488,256 (108.67 MB)

 Non-trainable params: 384 (1.50 KB)

In [1]:
import tensorflow as tf
import os

# GPU 메모리 할당 방식 변경 (필요한 만큼만 조금씩 할당)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth 설정 완료")
    except RuntimeError as e:
        print(e)


import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
import os
import glob
from tqdm import tqdm


# ==============================================================================
# 1. 설정 파라미터
# ==============================================================================
# 데이터 파일 경로 패턴
DATA_FILE_PATTERN = "channel_history_burst_mimo_3bs_final.npy"

# 물리 계층 파라미터
BANDWIDTH = 15e6
NUM_SUBCARRIERS = 64
FREQUENCIES = np.linspace(-BANDWIDTH/2, BANDWIDTH/2, NUM_SUBCARRIERS)

# 학습 파라미터
INPUT_SEQ_LEN = 10   # 과거 10개 보고
PRED_SEQ_LEN = 10    # 미래 10개 예측
BATCH_SIZE = 64      # GPU 메모리에 따라 조절 (32 ~ 128)
EPOCHS = 100         # 학습 반복 횟수
VALIDATION_SPLIT = 0.2 # 검증 데이터 비율

# ==============================================================================
# 2. 모델 정의 (보내주신 코드 + 수정)
# ==============================================================================
def build_paper_cnn_lstm(input_steps=10, pred_steps=10):
    # 입력: (Time, Rx=8, Tx=24, Freq_RealImag=128)
    inputs = Input(shape=(input_steps, 8, 24, 128))
    
    # 1. Spatial Feature Extraction (CNN)
    x = layers.TimeDistributed(layers.Conv2D(64, (3,3), padding='same', activation='relu'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.Conv2D(128, (3,3), padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    
    # Pooling: (8, 24) -> (4, 12)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)
    
    # Flatten: (Time, 4*12*128)
    x = layers.TimeDistributed(layers.Flatten())(x)
    
    # 2. Temporal Feature Extraction (LSTM)
    x = layers.LSTM(512, return_sequences=False)(x) # Encoder
    
    x = layers.RepeatVector(pred_steps)(x)            # Future placeholder
    x = layers.LSTM(512, return_sequences=True)(x)    # Decoder
    
    # 3. Output Reconstruction
    # 원래 차원인 8*24*128 (Rx*Tx*Freq*2)로 복원
    output_dim = 8 * 24 * 128
    x = layers.TimeDistributed(layers.Dense(output_dim))(x)
    outputs = layers.Reshape((pred_steps, 8, 24, 128))(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Paper_CNN_LSTM")
    return model

# ==============================================================================
# 3. 데이터 로드 및 전처리 (Physics -> AI Data)
# ==============================================================================
def load_and_preprocess_data():
    file_list = glob.glob(DATA_FILE_PATTERN)
    if not file_list:
        raise FileNotFoundError("❌ 데이터 파일(*.npy)을 찾을 수 없습니다. 시뮬레이션을 먼저 실행하세요.")
    
    print(f"📂 발견된 데이터 파일: {len(file_list)}개")
    
    all_bursts_H = [] # 변환된 H 행렬들을 담을 리스트

    for fname in file_list:
        print(f"Reading {fname}...")
        data = np.load(fname, allow_pickle=True)
        
        for burst in tqdm(data, desc=f"Processing {fname}"):
            a_list = burst['a']     # (30, 1, 8, 3, 8, Paths)
            tau_list = burst['tau'] # (30, 1, 3, Paths) -> Sionna 구조에 따라 다름
            
            # Burst 내 30개 스냅샷 처리
            h_seq = []
            for i in range(len(a_list)):
                # 1. Path Gain (a) 처리
                # 예상 Shape: (1, Rx=8, Tx=3, TxAnt=8, Paths)
                # 목표 Shape: (Rx=8, TxTotal=24, Paths)
                if a_list[i].size == 0:
                    h_seq.append(np.zeros((8, 24, 64), dtype=np.complex64))
                    continue
                    
                a_raw = a_list[i] 
                # 차원 축소 및 병합: (1, 8, 3, 8, Paths) -> (8, 3, 8, Paths) -> (8, 24, Paths)
                # squeeze로 불필요한 차원 제거 (Batch 등)
                try:
                    a_sq = a_raw.reshape(8, 3, 8, -1) 
                    a_flat = a_sq.reshape(8, 24, -1) # (Rx, Tx*TxAnt, Paths)
                except:
                    # Shape이 안 맞을 경우 예외 처리 (빈 데이터 등)
                    h_seq.append(np.zeros((8, 24, 64), dtype=np.complex64))
                    continue

                # 2. Delay (tau) 처리
                tau_raw = tau_list[i]
                # Broadcasting을 위해 차원 맞추기
                # tau는 보통 (Batch, Tx, Paths) 형태임 -> (1, 3, Paths)
                # 이를 (8, 24, Paths)로 확장해야 함
                try:
                    tau_sq = tau_raw.flatten() # 일단 펼침
                    # Paths 개수가 a와 맞는지 확인 필요하지만, 여기선 Broadcasting 이용
                    # 가장 간단한 방법: tau를 (1, 1, Paths)로 보고 확장
                    # 하지만 정확히는 Tx별로 다르므로, (3, Paths) -> (24, Paths) -> (8, 24, Paths)
                    
                    # 간단화: tau shape의 마지막 차원이 Paths라고 가정
                    num_paths = a_flat.shape[-1]
                    tau_flat = tau_raw.reshape(-1) # 전체 다 펼치고
                    # 경로 개수에 맞춰 자르거나 확장 (Sionna 구조 특성상 복잡하므로 단순화)
                    
                    # [중요] Sionna RT 출력 구조상, tau는 (Tx, Paths) 혹은 (1, Paths)일 수 있음.
                    # 여기서는 계산 효율을 위해 a_flat에 맞는 차원으로 강제 확장
                    tau_broadcast = tau_raw.reshape(1, 1, -1) if tau_raw.ndim == 1 else tau_raw
                    # (Broadcasting은 numpy가 알아서 처리하도록 유도)
                    
                except:
                    h_seq.append(np.zeros((8, 24, 64), dtype=np.complex64))
                    continue

                # 3. Frequency Response 계산: H = sum( a * exp(-j2pi * tau * f) )
                # Frequencies: (1, 1, 1, 64)
                # a_flat: (8, 24, Paths, 1)
                # tau: (..., Paths, 1)
                
                f_reshaped = FREQUENCIES.reshape(1, 1, 1, -1)
                a_input = a_flat[..., np.newaxis] 
                
                # tau 차원 맞추기 (약식: 경로 수만 맞으면 작동)
                if tau_raw.size > 0:
                    # tau의 마지막 차원이 Paths라고 가정
                    tau_input = tau_raw.flatten()[:a_flat.shape[-1]] # 경로 수 맞춤
                    tau_input = tau_input.reshape(1, 1, -1, 1)
                    
                    phase = -1j * 2 * np.pi * tau_input * f_reshaped
                    h_val = np.sum(a_input * np.exp(phase), axis=-2) # Sum over paths
                else:
                    h_val = np.zeros((8, 24, 64), dtype=np.complex64)

                h_seq.append(h_val) # (8, 24, 64)

            all_bursts_H.append(np.array(h_seq)) # (30, 8, 24, 64)

    return all_bursts_H

def create_dataset(bursts_data):
    X, Y = [], []
    print("🔄 데이터셋(X, Y) 생성 중...")
    
    for burst in bursts_data:
        # burst shape: (30, 8, 24, 64) (Complex)
        if len(burst) < INPUT_SEQ_LEN + PRED_SEQ_LEN:
            continue
            
        # 복소수 -> 실수/허수 채널 분리 (8, 24, 64) -> (8, 24, 128)
        # Real: [..., 0:64], Imag: [..., 64:128]
        burst_real = burst.real
        burst_imag = burst.imag
        burst_concat = np.concatenate([burst_real, burst_imag], axis=-1) # (30, 8, 24, 128)
        
        # 슬라이딩 윈도우
        for i in range(len(burst) - INPUT_SEQ_LEN - PRED_SEQ_LEN + 1):
            X.append(burst_concat[i : i+INPUT_SEQ_LEN])
            Y.append(burst_concat[i+INPUT_SEQ_LEN : i+INPUT_SEQ_LEN+PRED_SEQ_LEN])
            
    return np.array(X), np.array(Y)

# ==============================================================================
# 4. 메인 실행 (학습)
# ==============================================================================
if __name__ == "__main__":
    # 1. GPU 확인
    strategy = tf.distribute.MirroredStrategy()
    print(f"✅ 사용 가능한 GPU 수: {strategy.num_replicas_in_sync}")

    # 2. 데이터 로드 및 변환
    bursts = load_and_preprocess_data()
    if len(bursts) == 0:
        print("❌ 데이터가 비어있습니다.")
        exit()
        
    # 3. 학습 데이터셋 만들기
    X_train, Y_train = create_dataset(bursts)
    print(f"📊 학습 데이터 준비 완료:")
    print(f"   - X shape: {X_train.shape} (Samples, Time, Rx, Tx, Feat)")
    print(f"   - Y shape: {Y_train.shape}")
    
    # 4. 모델 생성 및 학습
    with strategy.scope():
        model = build_paper_cnn_lstm(INPUT_SEQ_LEN, PRED_SEQ_LEN)
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        
        print("\n🚀 학습 시작 (Training)...")
        history = model.fit(
            X_train, Y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=VALIDATION_SPLIT,
            verbose=1
        )
        
    # 5. 모델 저장
    model.save("cnn_lstm_mimo_model.h5")
    print("💾 모델 저장 완료: cnn_lstm_mimo_model.h5")

2025-12-31 11:16:39.759170: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767147399.772387  436155 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767147399.776407  436155 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767147399.787333  436155 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767147399.787343  436155 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767147399.787344  436155 computation_placer.cc:177] computation placer alr

✅ GPU Memory Growth 설정 완료
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1767147402.232123  436155 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 503 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1767147402.233492  436155 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 21796 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:c1:00.0, compute capability: 8.6


✅ 사용 가능한 GPU 수: 2
📂 발견된 데이터 파일: 1개
Reading channel_history_burst_mimo_3bs_final.npy...


Processing channel_history_burst_mimo_3bs_final.npy: 100%|██████████| 1000/1000 [00:06<00:00, 154.31it/s]


🔄 데이터셋(X, Y) 생성 중...
📊 학습 데이터 준비 완료:
   - X shape: (11000, 10, 8, 24, 128) (Samples, Time, Rx, Tx, Feat)
   - Y shape: (11000, 10, 8, 24, 128)

🚀 학습 시작 (Training)...


2025-12-31 11:17:17.738369: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 8.06GiB (rounded to 8650752000)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-12-31 11:17:17.738408: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2025-12-31 11:17:17.738421: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 28, Chunks in use: 28. 7.0KiB allocated for chunks. 7.0KiB in use in bin. 1.4KiB client-requested in use in bin.
2025-12-31 11:17:17.738429: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (512): 	Total Chunks: 5, Chunks in use: 5. 2.5KiB allocated for chunks. 2.5KiB in use in bin. 2.5KiB client-requested in use in bin.
2025-12-3

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input, mixed_precision
import os
import glob
from tqdm import tqdm
import gc

print("\n" + "="*60)
print("🚀 [논문 구현] GIE & Meta-Learning 기반 학습 코드 실행 (Fixed)")
print("   - 수정사항: KerasTensor 연산 오류 해결 (Lambda Layer 적용)")
print("="*60 + "\n")

# ==============================================================================
# 0. GPU 및 메모리 설정
# ==============================================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU Memory Growth 설정 완료 (GPU {len(gpus)}대)")
    except RuntimeError as e:
        print(e)

policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# ==============================================================================
# 1. 설정 파라미터
# ==============================================================================
DATA_FILE_PATTERN = "channel_history_burst_mimo_3bs_final*.npy"
BANDWIDTH = 15e6
NUM_SUBCARRIERS = 64
FREQUENCIES = np.linspace(-BANDWIDTH/2, BANDWIDTH/2, NUM_SUBCARRIERS)

INPUT_SEQ_LEN = 10   
PRED_SEQ_LEN = 10    
BATCH_SIZE = 16
EPOCHS = 50           
META_ITERATIONS = 3   

# ==============================================================================
# 2. 데이터 전처리
# ==============================================================================
def load_and_preprocess_data():
    file_list = glob.glob(DATA_FILE_PATTERN)
    if not file_list:
        raise FileNotFoundError("❌ 데이터 파일을 찾을 수 없습니다. 시뮬레이션을 먼저 실행하세요.")
    
    print(f"📂 데이터 파일 로드 중: {len(file_list)}개")
    all_bursts_H = [] 

    for fname in file_list:
        data = np.load(fname, allow_pickle=True)
        for burst in tqdm(data, desc=f"Processing {fname}"):
            a_list = burst['a']
            tau_list = burst['tau']
            
            h_seq = []
            for i in range(len(a_list)):
                if a_list[i].size == 0:
                    h_seq.append(np.zeros((8, 24, 64), dtype=np.complex64))
                    continue
                
                try:
                    a_raw = a_list[i]
                    tau_raw = tau_list[i]
                    a_sq = a_raw.reshape(8, 24, -1)
                    
                    f_reshaped = FREQUENCIES.reshape(1, 1, 1, -1).astype(np.float32)
                    tau_flat = tau_raw.flatten()[:a_sq.shape[-1]]
                    tau_input = tau_flat.reshape(1, 1, -1, 1).astype(np.float32)
                    a_input = a_sq[..., np.newaxis]
                    
                    phase = -1j * 2 * np.pi * tau_input * f_reshaped
                    h_val = np.sum(a_input * np.exp(phase), axis=-2)
                except:
                    h_val = np.zeros((8, 24, 64), dtype=np.complex64)

                h_seq.append(h_val)
            all_bursts_H.append(np.array(h_seq))
        del data
        gc.collect()
    return all_bursts_H

def create_dataset(bursts_data):
    X, Y = [], []
    for burst in bursts_data:
        if len(burst) < INPUT_SEQ_LEN + PRED_SEQ_LEN: continue
        burst_concat = np.concatenate([burst.real, burst.imag], axis=-1)
        for i in range(len(burst) - INPUT_SEQ_LEN - PRED_SEQ_LEN + 1):
            X.append(burst_concat[i : i+INPUT_SEQ_LEN])
            Y.append(burst_concat[i+INPUT_SEQ_LEN : i+INPUT_SEQ_LEN+PRED_SEQ_LEN])
            
    X_np = np.array(X, dtype=np.float32)
    Y_np = np.array(Y, dtype=np.float32)
    del X, Y
    gc.collect()
    return X_np, Y_np

# ==============================================================================
# 3. [수정됨] GIE 모듈 (Lambda Layer 사용)
# ==============================================================================
def GIE_Module(inputs):
    """
    Keras Functional API 호환성을 위해 tf 연산을 Lambda 레이어로 감쌉니다.
    """
    
    # 1. 시간축 Shift 연산을 위한 Lambda 함수 정의
    def shift_time_axis(x):
        # x shape: (Batch, Time, Rx, Tx, Freq)
        # 마지막 타임스텝 제거
        x_sliced = x[:, :-1, :, :, :]
        # 앞쪽에 0으로 채워진 프레임 추가 (Padding)
        # Padding config: [[Batch], [Time], [Rx], [Tx], [Freq]]
        x_shifted = tf.pad(x_sliced, [[0,0], [1,0], [0,0], [0,0], [0,0]])
        return x_shifted

    # 2. Lambda 레이어로 그래프에 연산 추가
    shifted_inputs = layers.Lambda(shift_time_axis, name="Time_Shift_Op")(inputs)
    
    # 3. 차분(Gradient) 계산 (Subtract 레이어 사용 권장)
    gradients = layers.Subtract(name="Gradient_Calc")([inputs, shifted_inputs])

    # 4. Gradient Embedding (CNN)
    gie_feat = layers.TimeDistributed(
        layers.Conv2D(64, (3,3), padding='same', activation='relu'), 
        name='GIE_Conv'
    )(gradients)
    
    gie_feat = layers.TimeDistributed(layers.BatchNormalization())(gie_feat)
    
    return gie_feat

# ==============================================================================
# 4. 모델 정의 (GIE + CNN-LSTM)
# ==============================================================================
def build_gie_model(input_steps=10, pred_steps=10, name="Student_Model"):
    inputs = Input(shape=(input_steps, 8, 24, 128), name='CSI_Input')
    
    # [GIE 적용]
    gie_features = GIE_Module(inputs)
    
    # [Spatial CNN]
    x = layers.TimeDistributed(layers.Conv2D(64, (3,3), padding='same', activation='relu'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    
    # [Integration] 원본 정보 + GIE(변화량) 정보 결합
    x = layers.Concatenate(axis=-1)([x, gie_features])
    
    # 추가 처리
    x = layers.TimeDistributed(layers.Conv2D(128, (3,3), padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)
    x = layers.TimeDistributed(layers.Flatten())(x)
    
    # [Spatiotemporal Memory (LSTM)]
    x = layers.LSTM(512, return_sequences=False)(x)
    x = layers.RepeatVector(pred_steps)(x)
    x = layers.LSTM(512, return_sequences=True)(x)
    
    # [Output]
    output_dim = 8 * 24 * 128
    x = layers.TimeDistributed(layers.Dense(output_dim, dtype='float32'))(x)
    outputs = layers.Reshape((pred_steps, 8, 24, 128))(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name=name)

# ==============================================================================
# 5. 메타 러닝 (Teacher-Student Loop) 실행
# ==============================================================================
if __name__ == "__main__":
    # 데이터 로드
    bursts = load_and_preprocess_data()
    X_full, Y_full = create_dataset(bursts)
    
    split_idx = int(len(X_full) * 0.5)
    X_labeled, Y_labeled = X_full[:split_idx], Y_full[:split_idx]
    X_unlabeled = X_full[split_idx:] 
    Y_unlabeled_true = Y_full[split_idx:] 
    
    print(f"📊 데이터 분할 완료: Labeled {len(X_labeled)} / Unlabeled {len(X_unlabeled)}")

    strategy = tf.distribute.MirroredStrategy()
    with strategy.scope():
        teacher_model = build_gie_model(name="Teacher")
        student_model = build_gie_model(name="Student")
        
        teacher_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        student_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # 메타 러닝 루프
    for iteration in range(META_ITERATIONS):
        print("\n" + "#"*60)
        print(f"🔄 Meta-Learning Iteration {iteration+1}/{META_ITERATIONS}")
        print("#"*60)
        
        print("👨‍🏫 [Teacher] 학습 중...")
        teacher_model.fit(X_labeled, Y_labeled, batch_size=BATCH_SIZE, epochs=5, verbose=1)
        
        print("🔮 [Teacher] Pseudo-Label 생성 중...")
        pseudo_labels = teacher_model.predict(X_unlabeled, batch_size=BATCH_SIZE)
        
        X_combined = np.concatenate([X_labeled, X_unlabeled], axis=0)
        Y_combined = np.concatenate([Y_labeled, pseudo_labels], axis=0)
        
        print(f"🧑‍🎓 [Student] 학습 중... (데이터 {len(X_combined)}개)")
        student_model.fit(X_combined, Y_combined, batch_size=BATCH_SIZE, epochs=5, verbose=1)
        
        print("💡 Evolving Teacher with Student's knowledge...")
        alpha = 0.5
        for t_var, s_var in zip(teacher_model.trainable_variables, student_model.trainable_variables):
            t_var.assign(alpha * t_var + (1 - alpha) * s_var)

    print("\n💾 최종 Student 모델 저장 중...")
    student_model.save("gie_meta_student_model.h5")
    
    print("\n🏆 최종 성능 평가 (Student Model on Unseen Data)")
    loss, mae = student_model.evaluate(X_unlabeled, Y_unlabeled_true, batch_size=BATCH_SIZE)
    print(f"   - Final MSE: {loss:.6f}")
    print(f"   - Final MAE: {mae:.6f}")


🚀 [논문 구현] GIE & Meta-Learning 기반 학습 코드 실행 (Fixed)
   - 수정사항: KerasTensor 연산 오류 해결 (Lambda Layer 적용)

✅ GPU Memory Growth 설정 완료 (GPU 2대)
📂 데이터 파일 로드 중: 1개


Processing channel_history_burst_mimo_3bs_final.npy: 100%|██████████| 1000/1000 [00:05<00:00, 178.72it/s]


📊 데이터 분할 완료: Labeled 5500 / Unlabeled 5500
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')

############################################################
🔄 Meta-Learning Iteration 1/3
############################################################
👨‍🏫 [Teacher] 학습 중...
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 th

I0000 00:00:1767163027.878922  449462 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1767163027.878928  449469 cuda_dnn.cc:529] Loaded cuDNN version 90300


344/344 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 2.9424e-10 - mae: 1.0841e-05

...

344/344 ━━━━━━━━━━━━━━━━━━━━ 20s 59ms/step - loss: 2.7044e-10 - mae: 1.0425e-05
Epoch 4/5
344/344 ━━━━━━━━━━━━━━━━━━━━ 20s 59ms/step - loss: 2.7069e-10 - mae: 1.0434e-05
Epoch 5/5


2025-12-31 15:38:30.655846: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
	 [[RemoteCall]]


344/344 ━━━━━━━━━━━━━━━━━━━━ 20s 59ms/step - loss: 2.7113e-10 - mae: 1.0440e-05
🔮 [Teacher] Pseudo-Label 생성 중...
344/344 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step
🧑‍🎓 [Student] 학습 중... (데이터 11000개)


2025-12-31 15:39:39.269086: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.
2025-12-31 15:40:01.270387: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 10.07GiB (rounded to 10813440000)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-12-31 15:40:01.270429: I ext

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.